# PhishNet-Transformer — Step 1: Dataset Preparation

**Goal of this notebook:** take the raw PhiUSIIL dataset and turn it into clean, balanced, correctly-labeled `train.csv`, `val.csv`, and `test.csv` files that both your classic ML model and your DistilBERT model will use later.

**Dataset:** PhiUSIIL Phishing URL Dataset
Download link: https://www.kaggle.com/datasets/ndarvind/phiusiil-phishing-url-dataset

After downloading, place the CSV file in the same folder as this notebook and update `DATA_PATH` in the next cell if the filename is different.

**Important label note:** in the raw PhiUSIIL file, `label = 1` means **legitimate** and `label = 0` means **phishing** — this is the opposite of the usual convention. We fix this in Cell 3 so that everywhere from here on, **1 = phishing, 0 = legitimate**, which is easier to reason about and matches what most tutorials/papers assume.

## Cell 1 — Import libraries and load the raw CSV

**Why:** before touching anything, we need to load the file and see its actual shape and columns with our own eyes — never assume a dataset looks the way documentation says it does.

**What we're doing:** reading the CSV with pandas and printing its shape (rows, columns) and column names.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

DATA_PATH = "PhiUSIIL_Phishing_URL_Dataset.csv"  # <-- update this if your downloaded filename differs

df = pd.read_csv(DATA_PATH)

print("Shape (rows, columns):", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
df.head()

**What we expect to see:** roughly 235,000 rows and 56 columns (the real PhiUSIIL dataset has many pre-extracted features like `URLLength`, `IsDomainIP`, etc.). We only need two of those 56 columns for this project — the raw `URL` text and the `label` — so the next cell trims it down.

## Cell 2 — Keep only the columns we need

**Why:** our project is specifically about learning from raw URL *text* (for the transformer) and comparing it against a classic model. We don't need the 50+ pre-extracted engineering features PhiUSIIL ships with for this notebook — keeping just `URL` and `label` keeps the data simple and keeps us focused on what we're actually building.

*(Note: if later you want your classic ML model to use PhiUSIIL's own pre-built features instead of building your own from scratch, you can come back to this cell and keep more columns. For now, we keep it minimal on purpose.)*

In [ ]:
data = df[['URL', 'label']].copy()
print("New shape:", data.shape)
data.head()

## Cell 3 — Fix the label convention

**Why:** as noted at the top, PhiUSIIL uses `1 = legitimate, 0 = phishing`. Every model, metric, and interview explanation is much easier if we standardize to the common convention: **1 = phishing (the positive/interesting class), 0 = legitimate**. We flip it once, here, so we never have to think about it again.

**What we're doing:** creating a new `label` column that is `1 - label_raw`, which flips 0s and 1s.

In [ ]:
data = data.rename(columns={'label': 'label_raw'})
data['label'] = 1 - data['label_raw']   # flip: now 1 = phishing, 0 = legitimate
data = data.drop(columns=['label_raw'])

print("Label counts after fixing convention (1 = phishing, 0 = legitimate):")
print(data['label'].value_counts())

**Interpretation:** you should now see roughly 100,945 rows labeled `1` (phishing) and 134,850 labeled `0` (legitimate) — matching the dataset's documented composition, just relabeled into the convention we'll use for the rest of the project.

## Cell 4 — Clean: drop nulls and duplicate URLs

**Why:** two common data quality issues in scraped/aggregated datasets like this are (a) missing values and (b) the exact same URL appearing more than once. Duplicates are a subtle danger — if the same URL ends up in both your train and test sets, your test accuracy becomes artificially inflated because the model has technically already "seen" that exact example. We remove both issues before splitting.

In [ ]:
before = len(data)

data = data.dropna(subset=['URL', 'label'])
data = data.drop_duplicates(subset=['URL'])

after = len(data)
print(f"Rows before cleaning: {before}")
print(f"Rows after cleaning:  {after}")
print(f"Dropped: {before - after} rows (nulls and/or duplicate URLs)")

## Cell 5 — Subsample to a balanced, manageable size

**Why:** the full dataset has ~235,000 rows. We don't need that many — a smaller, clean, balanced sample is easier to train on quickly (especially on a free Colab GPU), easier to reason about, and just as valid for demonstrating the comparison we care about. We also balance the two classes exactly 50/50, so accuracy is a meaningful metric later (with imbalanced classes, a model could get high accuracy just by always predicting the majority class — balancing avoids that trap).

**What we're doing:** randomly sampling 6,000 phishing and 6,000 legitimate URLs (12,000 total), using a fixed `random_state` so the sample is reproducible every time you rerun this notebook.

In [ ]:
N_PER_CLASS = 6000  # adjust down (e.g. 2000) if you want faster experimentation first

phishing = data[data['label'] == 1].sample(n=min(N_PER_CLASS, (data['label']==1).sum()), random_state=42)
legit    = data[data['label'] == 0].sample(n=min(N_PER_CLASS, (data['label']==0).sum()), random_state=42)

balanced = pd.concat([phishing, legit]).sample(frac=1, random_state=42).reset_index(drop=True)

print("Balanced dataset shape:", balanced.shape)
print(balanced['label'].value_counts())
balanced.head()

**Interpretation:** you should see exactly 6,000 rows for each class (12,000 total), shuffled together (`sample(frac=1)` shuffles the whole dataframe). This is now our full working dataset for the rest of the project.

## Cell 6 — Split into train / validation / test

**Why:** we need three separate, non-overlapping sets:
- **train** — what both models actually learn from
- **validation** — used during DistilBERT fine-tuning to check for overfitting (not shown to the model as training signal)
- **test** — touched only once, at the very end, to report your final honest comparison numbers

**Why `stratify=balanced['label']`:** this ensures each split keeps the same 50/50 phishing/legitimate ratio as the full dataset. Without it, random chance could give you a test set that's accidentally 70/30, which would make your evaluation numbers misleading.

In [ ]:
# First split off 70% train, 30% temp
train, temp = train_test_split(
    balanced, test_size=0.30, stratify=balanced['label'], random_state=42
)

# Then split temp evenly into validation (15%) and test (15%)
val, test = train_test_split(
    temp, test_size=0.50, stratify=temp['label'], random_state=42
)

print("Train:", train.shape)
print("Val:  ", val.shape)
print("Test: ", test.shape)

print("\nClass balance check (should all be close to 0.5 / 0.5):")
print("Train:\n", train['label'].value_counts(normalize=True))
print("Val:\n", val['label'].value_counts(normalize=True))
print("Test:\n", test['label'].value_counts(normalize=True))

**Interpretation:** with 12,000 total rows, this gives roughly 8,400 train / 1,800 validation / 1,800 test. All three splits should show close to a 50/50 label balance — if any split looks noticeably off, double check the `stratify` argument was applied correctly.

## Cell 7 — Sanity check: eyeball real examples

**Why:** metrics and shapes can look perfect while the underlying data is still wrong (wrong column mapped, labels flipped incorrectly, encoding issues, etc). The fastest way to catch this is to just look at a few real rows and confirm they make sense to a human.

In [ ]:
print("Sample PHISHING URLs (label = 1):")
print(train[train['label'] == 1]['URL'].head(5).to_string(index=False))

print("\nSample LEGITIMATE URLs (label = 0):")
print(train[train['label'] == 0]['URL'].head(5).to_string(index=False))

**What to look for:** the phishing examples should generally look suspicious to your eye — odd subdomains, unusual TLDs, long strings, misspelled brand names, etc. The legitimate ones should look like ordinary, recognizable web addresses. If the two lists look mixed up or indistinguishable, stop and re-check Cell 3's label flip before going further — this is the single most important checkpoint in this whole notebook.

## Cell 8 — Save the splits to disk

**Why:** saving these as their own CSV files means every later notebook (classic ML training, DistilBERT fine-tuning, evaluation) can just load `train.csv` / `val.csv` / `test.csv` directly, guaranteeing every model is trained and evaluated on the *exact same* split — which is what makes the final comparison between them fair and valid.

In [ ]:
train.to_csv("train.csv", index=False)
val.to_csv("val.csv", index=False)
test.to_csv("test.csv", index=False)

print("Saved train.csv, val.csv, test.csv to the current folder.")
print("\nFinal summary:")
print(f"  Train: {len(train)} rows")
print(f"  Val:   {len(val)} rows")
print(f"  Test:  {len(test)} rows")

## Step 1 complete — what we have now

- A clean, deduplicated, correctly-labeled dataset (1 = phishing, 0 = legitimate)
- Balanced 50/50 between the two classes
- Split into train / validation / test with matching class ratios in each
- Saved as three CSV files that every later notebook in this project will load

**Next step (Step 2):** re-run/confirm your existing classic ML model (XGBoost on lexical features) using this exact `train.csv` / `test.csv` split, so its reported accuracy is directly comparable to DistilBERT later.